<a href="https://colab.research.google.com/github/jeevithajeevitha3420-create/Qskill-Projects/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Task 2: Linear Regression Model to Predict House Price
--------------------------------------------------------
Loads house_data.csv, preprocesses features (encodes the categorical
Location column, scales numeric features), trains a Linear Regression
model, evaluates it, and visualizes predictions vs actual prices.

Run:  python3 house_price_regression.py
Requires: pandas, numpy, matplotlib, scikit-learn
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV_PATH = "house_data.csv"

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
df = pd.read_csv(CSV_PATH)
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(df.head())
print("\nShape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

# ---------------------------------------------------------------
# 2. Preprocessing
# ---------------------------------------------------------------
target = "Price"
numeric_features = ["Size_sqft", "Rooms", "Bathrooms", "Age_years",
                     "Garage_spaces", "Distance_to_city_km"]
categorical_features = ["Location"]

X = df[numeric_features + categorical_features]
y = df[target]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", LinearRegression()),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------------
# 3. Train
# ---------------------------------------------------------------
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# ---------------------------------------------------------------
# 4. Evaluate
# ---------------------------------------------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE (on held-out test set)")
print("=" * 60)
print(f"R-squared (R2)             : {r2:.4f}")
print(f"Mean Absolute Error (MAE)  : ${mae:,.2f}")
print(f"Root Mean Sq. Error (RMSE) : ${rmse:,.2f}")

# ---------------------------------------------------------------
# 5. Inspect coefficients (feature importance direction)
# ---------------------------------------------------------------
feature_names = (
    numeric_features
    + list(model.named_steps["preprocess"]
           .named_transformers_["cat"]
           .get_feature_names_out(categorical_features))
)
coefs = model.named_steps["regressor"].coef_
coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coefs})
coef_df = coef_df.sort_values("Coefficient", key=abs, ascending=False)
print("\nStandardized coefficients (impact on price, larger magnitude = bigger effect):")
print(coef_df.to_string(index=False))

# ---------------------------------------------------------------
# 6. Example prediction on a new house
# ---------------------------------------------------------------
sample = pd.DataFrame([{
    "Size_sqft": 2000, "Rooms": 4, "Bathrooms": 2, "Age_years": 10,
    "Garage_spaces": 2, "Distance_to_city_km": 5.0, "Location": "Suburb",
}])
sample_pred = model.predict(sample)[0]
print(f"\nExample: a 2000 sqft, 4-room, 2-bath, 10-yr-old Suburb house "
      f"with a 2-car garage, 5km from city center")
print(f"Predicted price: ${sample_pred:,.2f}")

# ---------------------------------------------------------------
# 7. Visualization 1: Predicted vs Actual
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.6, color="#4C72B0", edgecolor="k", linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("Actual Price ($)")
ax.set_ylabel("Predicted Price ($)")
ax.set_title(f"Predicted vs Actual House Price (R2 = {r2:.3f})", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("chart_predicted_vs_actual.png", dpi=150)
plt.close()
print("\nSaved: chart_predicted_vs_actual.png")

# ---------------------------------------------------------------
# 8. Visualization 2: Residuals
# ---------------------------------------------------------------
residuals = y_test - y_pred
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_pred, residuals, alpha=0.6, color="#DD8452", edgecolor="k", linewidth=0.3)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Predicted Price ($)")
ax.set_ylabel("Residual (Actual - Predicted)")
ax.set_title("Residual Plot", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("chart_residuals.png", dpi=150)
plt.close()
print("Saved: chart_residuals.png")

# ---------------------------------------------------------------
# 9. Visualization 3: Feature coefficient bar chart
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ["#55A868" if c > 0 else "#C44E52" for c in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors_bar, edgecolor="black")
ax.set_xlabel("Standardized Coefficient (effect on price)")
ax.set_title("Feature Impact on House Price", fontsize=13, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("chart_feature_importance.png", dpi=150)
plt.close()
print("Saved: chart_feature_importance.png")

DATASET OVERVIEW
   Size_sqft  Rooms  Bathrooms  Age_years  Garage_spaces  Distance_to_city_km  \
0       2018      5          4         52              2                 1.79   
1       1861      5          3         33              1                 2.21   
2       2347      7          4         19              1                11.05   
3       1375      4          2         53              1                10.68   
4       1769      5          2         53              1                 1.20   

   Location     Price  
0     Rural  195000.0  
1    Uptown  282900.0  
2    Uptown  416500.0  
3  Downtown  203600.0  
4    Suburb  202800.0  

Shape: (600, 8)

Missing values:
 Size_sqft              0
Rooms                  0
Bathrooms              0
Age_years              0
Garage_spaces          0
Distance_to_city_km    0
Location               0
Price                  0
dtype: int64

MODEL PERFORMANCE (on held-out test set)
R-squared (R2)             : 0.9469
Mean Absolute Error (MAE) 